# 02 - Baseline: TF-IDF + regresja logistyczna

Klasyczny, interpretowalny punkt odniesienia dla docelowego modelu **BiLSTM + Attention**. Pracuje na tej samej kolumnie `text` i konwencji etykiet (1 = Real, 0 = Fake) co reszta projektu, czytając gotowe splity CSV z `data/processed/splited/`.

Plan notatnika:
1. Trening pipeline'u TF-IDF + LogReg na zbiorze treningowym, dobór `C` na zbiorze walidacyjnym.
2. Metryki na walidacji i teście (Accuracy, Precision, Recall, F1, ROC-AUC), macierz pomyłek, krzywa ROC.
3. Wyjaśnialność — n-gramy o największym wpływie na predykcję (współczynniki regresji logistycznej).

Logika modelu jest wydzielona do `src/model/baseline.py` (`build_baseline`, `evaluate`, `top_features`).

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import roc_curve

from src.paths import DATA_SPLITS, REPO_ROOT
from src.model.baseline import build_baseline, evaluate, top_features

pd.options.plotting.backend = "plotly"
LABEL_COLORS = {"Fake": "#EF553B", "Real": "#636EFA"}

# Katalog na wyeksportowane wykresy (współdzielony z raportem i prezentacją).
FIGURES = REPO_ROOT / "docs" / "raport" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_SPLITS / "train.csv")
val = pd.read_csv(DATA_SPLITS / "val.csv")
test = pd.read_csv(DATA_SPLITS / "test.csv")

sizes_df = pd.DataFrame(
    {"split": ["train", "val", "test"], "liczba": [len(train), len(val), len(test)]}
)
sizes_df["udział_%"] = (sizes_df["liczba"] / sizes_df["liczba"].sum() * 100).round(1)
sizes_df

,split,liczba,udział_%
0,train,23082,60.0
1,val,7694,20.0
2,test,7694,20.0


## 1. Trening i dobór regularyzacji `C`

Trenujemy na zbiorze treningowym, a siłę regularyzacji `C` wybieramy po F1 na zbiorze walidacyjnym — zbiór testowy pozostaje nietknięty aż do raportu końcowego.

In [2]:
search = []
for C in [0.5, 1.0, 2.0, 4.0]:
    pipe = build_baseline(C=C)
    pipe.fit(train["text"], train["label"])
    m = evaluate(pipe, val["text"], val["label"])
    search.append({"C": C, **{k: m[k] for k in ["accuracy", "f1", "roc_auc"]}})

search_df = pd.DataFrame(search)
best_C = float(search_df.loc[search_df["f1"].idxmax(), "C"])

model = build_baseline(C=best_C)
model.fit(train["text"], train["label"])

search_df.style.hide(axis="index").highlight_max(
    subset=["accuracy", "f1", "roc_auc"], color="#cdebc5"
).set_caption(f"Dobór C na walidacji — wybrane C (wg F1) = {best_C}")

C,accuracy,f1,roc_auc
0.500000,0.981300,0.983100,0.998100
1.000000,0.983100,0.984700,0.998400
2.000000,0.987700,0.988800,0.999000
4.000000,0.990100,0.991100,0.999300


## 2. Metryki na walidacji i teście

In [3]:
val_metrics = evaluate(model, val["text"], val["label"])
test_metrics = evaluate(model, test["text"], test["label"])

metric_keys = ["accuracy", "precision", "recall", "f1", "roc_auc"]
metrics_df = pd.DataFrame(
    {
        "Metryka": metric_keys,
        "Walidacja": [val_metrics[k] for k in metric_keys],
        "Test": [test_metrics[k] for k in metric_keys],
    }
)
metrics_df

,Metryka,Walidacja,Test
0,accuracy,0.9901,0.9875
1,precision,0.9880,0.9857
2,recall,0.9941,0.9917
3,f1,0.9911,0.9887
4,roc_auc,0.9993,0.9986


In [4]:
cm = test_metrics["confusion_matrix"]
labels = ["Fake (0)", "Real (1)"]
fig = px.imshow(
    cm,
    x=labels,
    y=labels,
    text_auto=True,
    color_continuous_scale="Blues",
    labels={"x": "Predykcja", "y": "Prawdziwa klasa", "color": "liczba"},
    title="Macierz pomyłek — zbiór testowy",
)
fig.update_layout(height=420, coloraxis_showscale=False)
fig.write_image(FIGURES / "baseline_macierz_pomylek.png", scale=2)
fig.show()

In [5]:
y_proba = model.predict_proba(test["text"])[:, 1]
fpr, tpr, _ = roc_curve(test["label"], y_proba)
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=fpr, y=tpr, mode="lines", name=f"ROC (AUC={test_metrics['roc_auc']})")
)
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line=dict(dash="dash", color="gray"),
        name="losowy",
    )
)
fig.update_layout(
    title="Krzywa ROC — zbiór testowy",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=440,
)
fig.write_image(FIGURES / "baseline_roc.png", scale=2)
fig.show()

**Wniosek (1).** Baseline TF-IDF (1–2-gramy) + regresja logistyczna osiąga na zbiorze testowym bardzo wysokie F1 i ROC-AUC (~0.99 / ~0.999). To oczekiwane dla tego zbioru — nawet po usunięciu markerów wycieku klasy różnią się silnie leksykalnie i stylistycznie. Wysoki wynik baseline'u wyznacza **wymagający punkt odniesienia**: docelowy BiLSTM + Attention musi mu dorównać lub go przewyższyć, a jego główną wartością będzie raczej modelowanie kontekstu i wyjaśnialność przez wagi atencji niż sam wzrost metryk.

## 3. Wyjaśnialność — współczynniki regresji logistycznej

Regresja logistyczna jest z natury interpretowalna: znak i wielkość współczynnika przy każdym n-gramie mówi, jak silnie pcha on predykcję w stronę Real (dodatni) lub Fake (ujemny). `top_features` wyciąga n-gramy o największej sile dla każdej klasy.

In [6]:
real_feats, fake_feats = top_features(model, n=20)

feat_df = pd.DataFrame(
    [(t, c, "Real") for t, c in real_feats] + [(t, c, "Fake") for t, c in fake_feats],
    columns=["n-gram", "współczynnik", "klasa"],
).sort_values("współczynnik")

fig = px.bar(
    feat_df,
    x="współczynnik",
    y="n-gram",
    color="klasa",
    color_discrete_map=LABEL_COLORS,
    orientation="h",
    title="Najsilniejsze n-gramy wg współczynników regresji logistycznej",
)
fig.update_layout(
    height=760,
    xaxis_title="współczynnik LR (Real > 0, Fake < 0)",
    yaxis_title="",
    legend_title="",
)
fig.write_image(FIGURES / "logreg_cechy.png", scale=2)
fig.show()

**Wniosek (2).** Najsilniejsze cechy są interpretowalne i — co kluczowe — **nie są resztkowym wyciekiem**. Stronę Real ciągną zwroty agencyjno-sprawozdawcze (`said`, `on wednesday`, `president donald`), typowe dla języka depesz. Stronę Fake ciągną zwroty potoczne / clickbaitowe (`via`, `read more`, `you`, `just`). Żadne ze ścisłych markerów wyciętych w `03_cleaning.ipynb` (np. `reuters`, URL-e, `@handle`, `getty`) nie pojawia się na liście — to walidacja, że model uczy się różnic stylu i tematu, a nie stempla redakcyjnego. Per-przykładowa wyjaśnialność modelu neuronowego (wizualizacja wag atencji) jest zaplanowana na etap BiLSTM + Attention.

## 4. Podsumowanie

- Baseline **TF-IDF + regresja logistyczna** działa na tych samych splitach CSV i konwencji etykiet co reszta projektu; cała logika jest reużywalna w `src/model/baseline.py`.
- Wynik (F1 / AUC ~0.99 / ~0.999 na teście) stanowi punkt odniesienia dla docelowego BiLSTM + Attention.
- Współczynniki LR dają tanią wyjaśnialność i potwierdzają brak resztkowego wycieku po czyszczeniu.